[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/16_slos_alertas_incidentes/16_proyecto_slos_alertas_incidentes.ipynb)

# Proyecto 16 · SLOs, alertas, runbook y postmortem de SupportOps

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración estimada** | ≈ 1 h 45 min |
| **Servicios de Compose** | Ninguno (todo en memoria). Opcional: `core` + `metrics` para ver tus reglas en Grafana (paso final) |
| **RAM aproximada** | < 500 MB |
| **Prerrequisitos** | Lección `16_slos_alertas_incidentes`, proyectos 14 y 15 y módulo 10 |
| **Conexión** | Cierra el bloque de calidad: la telemetría de los módulos 9-15 se convierte en **contrato** (SLO), **alerta** y **aprendizaje** (postmortem) |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. Dirección pide cuatro cosas: (1) SLOs medibles para disponibilidad, latencia, calidad y coste, (2) alertas que despierten solo por lo que agota el presupuesto,
(3) las reglas de alerta provisionadas como código y (4) tras el incidente de ayer —un fallo del proveedor LLM entre las 10:00 y las 10:45— un **postmortem sin culpa** con cifras reales.

## Datos

Sintéticos y deterministas (semilla 42, `mock`): el **Paso 0** ejecuta el agente real sobre 24 h de tráfico (`simulate_day`) con el incidente del proveedor y una regresión de prompt. Recibes los eventos por ticket (`ROWS`) y los
contadores por minuto de disponibilidad (`AV_G`, `AV_T`) y calidad (`QU_G`, `QU_T`). La verdad del incidente (`TRUTH`) está en el notebook: úsala solo para validar.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución; `mock`, semilla 42) |
|---|---|---|---|
| R1 | `sli_counts` | Buenos/totales de los 4 SLIs sobre el día completo | exacto |
| R2 | `budget_math`, `hours_to_exhaustion` | Presupuesto, consumo, burn y tiempo hasta agotar en 5 vectores | 5/5 |
| R3 | `multiwindow_fire` | Igual que la referencia en disponibilidad y calidad; sin falsos positivos fuera del incidente | exacto / 3/3 |
| R4 | `alert_rule`, `provision_group` | Estructura de la regla, anotaciones, `for`, y rechazo de intervalos no múltiplos de 10 s | 7/7 |
| R5 | `incident_report` | TTD, reajuste, tickets perdidos y % del presupuesto mensual | exacto (tol. 1e-9) |
| R6 | `write_postmortem` | Secciones, cifras coherentes, ≥ 4 acciones con responsable/plazo/tipo y sin lenguaje de culpa | 7/7 |

> 📏 Los umbrales salen de la solución de referencia con la semilla 42 fija en el notebook; R1, R3 y R5 se comparan con las funciones del paquete `supportops.slo` ejecutadas en vivo.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
if str(ROOT) not in sys.path:   # un editable recién instalado no es visible hasta reiniciar el kernel: lo añadimos a mano
    sys.path.insert(0, str(ROOT))
import importlib; importlib.invalidate_caches()
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import collections, dataclasses, json, math, re, time, threading
import yaml
from IPython.display import Markdown
from supportops.slo import (BASELINE_FAULTS, DEFAULT_SLOS, GOOGLE_POLICIES, LAB_POLICIES, DayScenario, MultiWindowPolicy, alert_quality, bin_events, budget_report,
                            burn_promql, burn_series, fire_series, grafana_burn_rule, minute_mask, pages_per_week, render_postmortem, rules_yaml, simulate_day,
                            simulate_month, sli_promql, threshold_fire_series, trailing_sum)
STACK = check_stack([])
LIVE = False

## Paso 0 · Los datos (te lo damos hecho)

In [ ]:
DAY = simulate_day(SEED)
ROWS, NB, T0, TRUTH = DAY["rows"], DAY["n_bins"], DAY["t0"], DAY["truth"]
AV_G, AV_T = bin_events(ROWS, DEFAULT_SLOS["availability"], t0=T0, n_bins=NB)
QU_G, QU_T = bin_events(ROWS, DEFAULT_SLOS["quality"], t0=T0, n_bins=NB)
O0, O1 = (int(x) for x in TRUTH["outage"])
print(f"{len(ROWS)} tickets · incidente real: minutos {O0}-{O1} · fila de ejemplo: {ROWS[0]}")
print({k: (v.description, v.objective, v.threshold) for k, v in DEFAULT_SLOS.items()})

## Paso 1 · Los SLIs

**TODO 1.** `sli_counts(rows, slo)` recibe la lista de filas (dicts con `status`, `latency_s`, `cost_usd`, `quality_good`) y un `dict` `slo` con `sli` y `threshold`, y devuelve `(buenos, totales)`:

- `availability`: bueno si `status` ∈ {`ok`, `escalated`}; todos los tickets son válidos.
- `latency`: bueno si `latency_s <= threshold`.
- `cost`: bueno si `cost_usd <= threshold`.
- `quality`: **válidos** solo los que tienen `quality_good` distinto de `None` (los evaluados); bueno si `quality_good` es `True`.

<details>
<summary>🪜 Pista 1</summary>

Es el mismo patrón en los cuatro casos: construye una lista de booleanos «¿este evento es bueno?» sobre los **eventos válidos** y devuelve `(sum(lista), len(lista))`. Solo cambia qué filas son válidas (en calidad, las evaluadas).

</details>

In [ ]:
def sli_counts(rows: list, slo: dict) -> tuple:
    # TODO 1
    raise NotImplementedError("TODO 1")

try:
    print({k: sli_counts(ROWS, {"sli": v.sli, "threshold": v.threshold}) for k, v in DEFAULT_SLOS.items()})
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 2 · Presupuesto y burn rate

**TODO 2a.** `budget_math(good, total, objective)` devuelve `{"sli", "bad", "allowed_bad", "consumed", "burn"}`: `sli = good/total`, `bad = total − good`, `allowed_bad = (1 − objective)·total`, `consumed = bad/allowed_bad` y `burn = (1 − sli)/(1 − objective)`.

**TODO 2b.** `hours_to_exhaustion(burn, period_h=720)` devuelve las horas hasta agotar **todo** el presupuesto del periodo a ese ritmo: `period_h / burn` (`float("inf")` si `burn <= 0`).

In [ ]:
def budget_math(good: float, total: float, objective: float) -> dict:
    # TODO 2a
    raise NotImplementedError("TODO 2a")

def hours_to_exhaustion(burn: float, period_h: float = 720) -> float:
    # TODO 2b
    raise NotImplementedError("TODO 2b")

try:
    print(budget_math(9_900, 10_000, 0.99), hours_to_exhaustion(14.4))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 3 · La alerta multiventana

**TODO 3.** `multiwindow_fire(good, total, objective, factor, long_min, short_min, min_events=5)` recibe arrays por minuto y devuelve un array booleano (uno por minuto): `True` si el burn rate de la ventana **larga** y el de la **corta** son `>= factor`.

- Cada ventana suma buenos y totales de los **últimos** `w` minutos (incluido el actual; al principio, los que haya).
- Si en la ventana hay menos de `min_events` eventos, ese burn **no cuenta** (no dispara).
- `burn = (1 − buenos/totales)/(1 − objective)`.

<details>
<summary>🪜 Pista 1</summary>

Suma móvil con `np.cumsum`: `c = np.cumsum(np.insert(x, 0, 0))`; la suma de los últimos `w` hasta `i` es `c[i+1] - c[max(i+1-w, 0)]`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Escribe una función interna `burn(w)` que, para todos los minutos `i = np.arange(len(good))`, calcule `g` y `t` con esa resta, ponga `NaN`/0 donde `t < min_events` y devuelva `(1 - g/t)/(1 - objective)`. El resultado es `(burn(long_min) >= factor) & (burn(short_min) >= factor)`. Compáralo con tu `burn_by_hand` de la lección §4.

</details>

In [ ]:
def multiwindow_fire(good, total, objective, factor, long_min, short_min, min_events=5):
    # TODO 3
    raise NotImplementedError("TODO 3")

try:
    F = multiwindow_fire(AV_G, AV_T, 0.99, 14.4, 60, 5)
    print("dispara", int(F.sum()), "minutos · primer minuto:", int(np.argmax(F)) if F.any() else None, "· incidente real:", O0)
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 4 · La regla de Grafana como código

**TODO 4a.** `alert_rule(name, objective, threshold_sli, factor, long_w, short_w, severity, dashboard_uid, panel_id)` devuelve el `dict` de una regla de provisioning (formato del módulo 10) para un SLO de **disponibilidad**:

- `uid` = `f"slo-{name}-{severity}"`, `title` con el nombre, `condition` = `"F"`, `for` = `"0s"` si `severity == "page"` y `"1m"` si no, `noDataState` = `"OK"`, `execErrState` = `"Error"`.
- `data`: lista con refIds **A** (PromQL instantánea del burn con la ventana `long_w`, datasource `prometheus`), **B** (idem con `short_w`), **C** y **D** (`reduce` `last` con `settings.mode = dropNN` de A y B), **E** (`math`: `$C > f && $D > f`) y **F** (`threshold` `gt 0` sobre E). El burn de availability es `(1 - ratio) / (1 - objective)` con
  `ratio = sum(rate(supportops_agent_requests_total{service_name="supportops-agent",supportops_status=~"ok|escalated"}[W])) / sum(rate(supportops_agent_requests_total{service_name="supportops-agent"}[W]))`.
- `annotations`: `__dashboardUid__`, `__panelId__` (**cadena**), `summary` y `runbook_url`. `labels`: `severity`, `service="supportops-agent"`, `slo=name`.

**TODO 4b.** `provision_group(rules, interval)` devuelve `{"apiVersion": 1, "groups": [{"orgId": 1, "name": "supportops-slo", "folder": "Curso observabilidad de agentes", "interval": interval, "rules": rules}]}` y lanza `ValueError` si `interval` no termina en `s` o no es múltiplo de 10 s.

<details>
<summary>🪜 Pista 1</summary>

No inventes la estructura: es la misma regla que la lección imprimió en §7 (`r0["data"]` y la tabla de piezas A-F). Reprodúcela como un `dict` y comprueba con `[d["refId"] for d in R["data"]]`. Escribe antes una función pequeña `expr(w)` que devuelva el PromQL del burn para una ventana `w` y reutilízala para A y B.

</details>

<details>
<summary>🪜 Pista 2</summary>

`A` y `B` son consultas (`datasourceUid="prometheus"`, `model` con `expr` e `instant=True`); `C` y `D` son `reduce` con `reducer="last"` sobre A y B; `E` es un `math` con `$C > f && $D > f` y `F` un `threshold` `gt 0` sobre E. Los `dict` de `annotations` y `labels` salen del enunciado tal cual (`__panelId__` es `str(panel_id)`).

</details>

In [ ]:
def alert_rule(name, objective, threshold_sli, factor, long_w, short_w, severity, dashboard_uid, panel_id) -> dict:
    # TODO 4a
    raise NotImplementedError("TODO 4a")

def provision_group(rules, interval) -> dict:
    # TODO 4b
    raise NotImplementedError("TODO 4b")

try:
    R = alert_rule("availability", 0.99, None, 14.4, "5m", "1m", "page", "slo-agente", 12)
    print(R["uid"], [d["refId"] for d in R["data"]], R["annotations"]["__panelId__"])
    print(provision_group([R], "30s")["groups"][0]["interval"])
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 5 · Las cifras del incidente

**TODO 5.** `incident_report(good, total, fire, start_min, end_min, objective)` devuelve un `dict` con los arrays por minuto de disponibilidad, el array booleano `fire` de la alerta y la ventana real del incidente:

- `ttd_min`: minutos desde `start_min` hasta el **primer** minuto `>= start_min` con `fire` verdadero.
- `reset_min`: minutos desde `end_min` hasta el **último** minuto con `fire` verdadero (puede ser negativo si se apagó antes).
- `bad_tickets`: tickets malos (`total − good`) en `[start_min, end_min)` (entero).
- `budget_month_pct`: `100 · bad_tickets / ((1 − objective) · total.sum() · 30)` (el día se repite 30 veces).
- `false_minutes`: minutos con `fire` verdadero fuera de `[start_min − 5, end_min + 60)`.

<details>
<summary>🪜 Pista 1</summary>

Con `fire` booleano: el primer minuto que dispara dentro del incidente es `np.argmax(fire[start_min:]) `; el último minuto que dispara es `np.where(fire)[0].max()`. Los tickets malos son `int((total - good)[start_min:end_min].sum())`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`false_minutes`: `m = np.ones(len(fire), bool); m[max(start_min-5, 0):end_min+60] = False` y cuenta `fire & m`. Es la misma idea de «alertas fuera del incidente» que viste en la comparación de estrategias (§4).

</details>

In [ ]:
def incident_report(good, total, fire, start_min, end_min, objective) -> dict:
    # TODO 5
    raise NotImplementedError("TODO 5")

REPORT = {}
try:
    REPORT = incident_report(AV_G, AV_T, multiwindow_fire(AV_G, AV_T, 0.99, 14.4, 60, 5), O0, O1, 0.99)
    print(REPORT)
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 6 · El postmortem sin culpa

**TODO 6.** `write_postmortem(report)` devuelve el `dict` que acepta `render_postmortem` (claves `title, summary, impact, timeline, root_cause, contributing, went_well, went_wrong, lucky, actions`). Requisitos:

- `summary` e `impact` incluyen las **cifras de `report`**: el TTD (en minutos), los tickets malos y el % del presupuesto mensual (redondeado a entero).
- `timeline`: ≥ 5 entradas `("HH:MM", "evento")` en **orden cronológico** que incluyan el inicio real del incidente (`10:00`) y la hora de la alerta.
- `actions`: ≥ 4 tuplas `(acción, responsable, plazo, tipo)` con `tipo` ∈ {`prevenir`, `detectar`, `mitigar`}, **al menos una de cada tipo**, y responsables que sean equipos/roles (no nombres propios).
- **Sin culpa**: ni `root_cause` ni `contributing` contienen «culpa», «negligen», «error humano», «debería haber» ni «se olvidó».

<details>
<summary>🪜 Pista 1</summary>

Puedes reutilizar la estructura de la lección (§9). Para la hora de alerta usa `f'{(O0 + report["ttd_min"]) // 60:02d}:{(O0 + report["ttd_min"]) % 60:02d}'`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Parte del `pm` de la lección: cambia solo lo que depende de `report` (cifras del resumen y la hora de la alerta) y revisa que cada acción tenga la forma `(acción, equipo, plazo, tipo)` y que haya al menos una de `prevenir`, `detectar` y `mitigar`. Evita en causa raíz las palabras prohibidas: describe el **sistema** (un único proveedor, ausencia de reserva), no a una persona.

</details>

In [ ]:
def write_postmortem(report: dict) -> dict:
    # TODO 6
    raise NotImplementedError("TODO 6")

try:
    PM = write_postmortem(REPORT); print(render_postmortem(PM)[:600])
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 7 · Rúbrica

In [ ]:
def run_rubric():
    rows = []
    def add(r, d, ok, det=""): rows.append({"criterio": r, "descripción": d, "resultado": "✅ OK" if ok else "❌ FALLA", "detalle": det})
    # R1
    try:
        got = {k: tuple(int(x) for x in sli_counts(ROWS, {"sli": v.sli, "threshold": v.threshold})) for k, v in DEFAULT_SLOS.items()}
        ref = {k: (int(bin_events(ROWS, v, t0=T0, n_bins=NB)[0].sum()), int(bin_events(ROWS, v, t0=T0, n_bins=NB)[1].sum())) for k, v in DEFAULT_SLOS.items()}
        add("R1", "sli_counts: 4 SLIs", got == ref, f"{got}" if got != ref else f"exacto · {got}")
    except Exception as e: add("R1", "sli_counts", False, f"{type(e).__name__}: {e}")
    # R2
    try:
        b = budget_math(9_900, 10_000, 0.99); b2 = budget_math(9_900, 10_000, 0.999); b3 = budget_math(10_000, 10_000, 0.99)
        checks = {"consumo = 1": abs(b["consumed"] - 1) < 1e-9 and abs(b["burn"] - 1) < 1e-9, "allowed_bad": abs(b["allowed_bad"] - 100) < 1e-9 and b["bad"] == 100, "burn 10×": abs(b2["burn"] - 10) < 1e-9 and abs(b2["consumed"] - 10) < 1e-9,
                  "14,4 → 50 h": abs(hours_to_exhaustion(14.4) - 50) < 1e-9, "burn 0 → inf": hours_to_exhaustion(0) == float("inf") and b3["consumed"] == 0}
        add("R2", "budget_math y hours_to_exhaustion: 5 vectores", all(checks.values()), str({k: v for k, v in checks.items() if not v} or "todas OK"))
    except Exception as e: add("R2", "budget_math", False, f"{type(e).__name__}: {e}")
    # R3
    try:
        p = GOOGLE_POLICIES[0]
        f1 = multiwindow_fire(AV_G, AV_T, 0.99, 14.4, 60, 5); r1 = fire_series(AV_G, AV_T, 0.99, p)
        inc = minute_mask(O0, O1, NB)
        checks = {"disponibilidad = referencia": bool((np.asarray(f1) == r1).all()), "dispara dentro del incidente": bool(np.asarray(f1)[inc].any()), "sin falsos positivos fuera (±60 min)": not np.asarray(f1)[: O0 - 5].any() and not np.asarray(f1)[O1 + 60:].any()}
        add("R3", "multiwindow_fire: 3 comprobaciones", all(checks.values()), str({k: v for k, v in checks.items() if not v} or f"todas OK · primer minuto {int(np.argmax(f1))}"))
    except Exception as e: add("R3", "multiwindow_fire", False, f"{type(e).__name__}: {e}")
    # R4
    try:
        r = alert_rule("availability", 0.99, None, 14.4, "5m", "1m", "page", "slo-agente", 12)
        r2 = alert_rule("availability", 0.99, None, 3.0, "15m", "3m", "ticket", "slo-agente", 12)
        ref = grafana_burn_rule(DEFAULT_SLOS["availability"], LAB_POLICIES[0], dashboard_uid="slo-agente", panel_id=12)
        by = {d["refId"]: d for d in r["data"]}
        def raises(iv):
            try: provision_group([r], iv)
            except ValueError: return True
            return False
        checks = {"refIds A-F y condición F": [d["refId"] for d in r["data"]] == list("ABCDEF") and r["condition"] == "F", "burn largo y corto": by["A"]["model"]["expr"] == ref["data"][0]["model"]["expr"] and by["B"]["model"]["expr"] == ref["data"][1]["model"]["expr"],
                  "math y umbral": by["E"]["model"]["expression"].replace(" ", "") == "$C>14.4&&$D>14.4" and by["F"]["model"]["type"] == "threshold",
                  "for según severidad": r["for"] == "0s" and r2["for"] == "1m", "anotaciones": r["annotations"]["__dashboardUid__"] == "slo-agente" and r["annotations"]["__panelId__"] == "12" and "runbook_url" in r["annotations"] and "summary" in r["annotations"],
                  "etiquetas": r["labels"] == {"severity": "page", "service": "supportops-agent", "slo": "availability"} or (r["labels"].get("severity") == "page" and r["labels"].get("slo") == "availability"),
                  "intervalo múltiplo de 10 s": provision_group([r], "30s")["groups"][0]["interval"] == "30s" and raises("15s") and raises("45")}
        add("R4", "alert_rule y provision_group: 7 comprobaciones", all(checks.values()), str({k: v for k, v in checks.items() if not v} or "todas OK"))
    except Exception as e: add("R4", "alert_rule/provision_group", False, f"{type(e).__name__}: {e}")
    # R5
    try:
        f = multiwindow_fire(AV_G, AV_T, 0.99, 14.4, 60, 5); rep = incident_report(AV_G, AV_T, f, O0, O1, 0.99)
        idx = np.where(f)[0]; first = int(idx[idx >= O0][0]); last = int(idx[-1])
        exp = {"ttd_min": first - O0, "reset_min": last - O1, "bad_tickets": int((AV_T[O0:O1] - AV_G[O0:O1]).sum()), "budget_month_pct": 100 * (AV_T[O0:O1] - AV_G[O0:O1]).sum() / (0.01 * AV_T.sum() * 30),
               "false_minutes": int(f[: max(O0 - 5, 0)].sum() + f[O1 + 60:].sum())}
        ok = all(abs(rep[k] - v) < 1e-9 for k, v in exp.items())
        add("R5", "incident_report: 5 cifras", ok, f"{ {k: round(rep[k], 3) for k in exp} } (ref {exp})")
    except Exception as e: add("R5", "incident_report", False, f"{type(e).__name__}: {e}")
    # R6
    try:
        rep = incident_report(AV_G, AV_T, multiwindow_fire(AV_G, AV_T, 0.99, 14.4, 60, 5), O0, O1, 0.99); pm = write_postmortem(rep); md_ = render_postmortem(pm)
        txt = json.dumps(pm, ensure_ascii=False).lower()
        times = [h for h, _ in pm["timeline"]]
        hdet = f"{(O0 + rep['ttd_min']) // 60:02d}:{(O0 + rep['ttd_min']) % 60:02d}"
        types = {a[3] for a in pm["actions"]}
        blame = ["culpa", "negligen", "error humano", "debería haber", "se olvidó"]
        checks = {"secciones": all(h in md_ for h in ("## Resumen", "## Impacto", "## Cronología", "## Acciones")),
                  "cifras coherentes": str(int(rep["ttd_min"])) in pm["summary"] and str(rep["bad_tickets"]) in txt and str(round(rep["budget_month_pct"])) in txt,
                  "cronología ≥ 5, ordenada, con inicio y alerta": len(times) >= 5 and times == sorted(times) and "10:00" in times and hdet in times,
                  "≥ 4 acciones completas": len(pm["actions"]) >= 4 and all(len(a) == 4 and all(str(x).strip() for x in a) for a in pm["actions"]),
                  "los 3 tipos": {"prevenir", "detectar", "mitigar"} <= types,
                  "responsables = equipos": all(not re.search(r"\b[A-Z][a-z]+ [A-Z][a-z]+\b", a[1]) for a in pm["actions"]),
                  "sin lenguaje de culpa": not any(b in (pm["root_cause"] + " ".join(pm["contributing"])).lower() for b in blame)}
        add("R6", "write_postmortem: 7 comprobaciones", all(checks.values()), str({k: v for k, v in checks.items() if not v} or "todas OK"))
    except Exception as e: add("R6", "write_postmortem", False, f"{type(e).__name__}: {e}")
    return pd.DataFrame(rows).set_index("criterio")

In [ ]:
# Ejecuta esta celda cuando hayas terminado los TODO
display(run_rubric())

---
## Paso final (opcional) · Míralo funcionar

Con el stack: escribe `provision_group(...)` a `stack/grafana/provisioning/alerting/` con `yaml.safe_dump` y recarga con `POST /api/admin/provisioning/alerting/reload`; después reproduce el incidente en vivo con el código de la sección 7.1 de la lección.
Comprueba que tu regla de página pasa a *firing* y se resuelve sola.

## Retos extra (nivel experto)

1. **Cuatro SLOs, ocho reglas**: genera las reglas de los cuatro SLOs y de ambas políticas con un bucle y compara tu YAML con `16-slo-alerts.yaml`.
2. **Sensibilidad**: barre `factor` (3-30) y `long_min` (15-240) y dibuja TTD frente a minutos falsos con 40 meses simulados (`simulate_month`); elige el punto de operación y defiéndelo.
3. **Política escalonada**: implementa «<50 % de presupuesto → revisar despliegues; <25 % → solo fiabilidad; agotado → congelar» como función de `budget_math` sobre un mes simulado.
4. **SLO de un cliente premium** con 20 tickets/día: diseña ventanas, `min_events` y una sonda sintética que lo hagan alertable.
5. **Segundo postmortem**: escribe el de la regresión de prompt (6:00-14:00) con las cifras de calidad; ¿qué acciones de *prevenir* cambian respecto al del proveedor?

## Reflexión (producción e hilo conductor)

- Tu TTD es de varios minutos. ¿Qué cuesta acortarlo (falsas alarmas, sondas sintéticas) y quién decide el compromiso?
- ¿Qué SLO pondrías al proveedor del LLM en el contrato y cómo lo medirías tú, no él?
- En el capstone (módulo 22) el agente es multi-agente: ¿cómo descompones el SLO de extremo a extremo entre orquestador y subagentes?
- ¿Qué parte del postmortem sería distinta si el incidente hubiera sido una fuga de PII (módulo 20)?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable (redefine las funciones y vuelve a lanzar la rúbrica).

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
def sli_counts(rows, slo):
    k = slo["sli"]
    if k == "availability": ok = [r["status"] in ("ok", "escalated") for r in rows]
    elif k == "latency": ok = [r["latency_s"] <= slo["threshold"] for r in rows]
    elif k == "cost": ok = [r["cost_usd"] <= slo["threshold"] for r in rows]
    else: ok = [bool(r["quality_good"]) for r in rows if r["quality_good"] is not None]
    return sum(ok), len(ok)

def budget_math(good, total, objective):
    sli = good / total; bad = total - good; allowed = (1 - objective) * total
    return {"sli": sli, "bad": bad, "allowed_bad": allowed, "consumed": bad / allowed, "burn": (1 - sli) / (1 - objective)}

def hours_to_exhaustion(burn, period_h=720):
    return float("inf") if burn <= 0 else period_h / burn

def multiwindow_fire(good, total, objective, factor, long_min, short_min, min_events=5):
    cg, ct = np.cumsum(np.insert(np.asarray(good, float), 0, 0)), np.cumsum(np.insert(np.asarray(total, float), 0, 0))
    def burn(w):
        i = np.arange(len(good)); lo = np.maximum(i + 1 - w, 0)
        g, t = cg[i + 1] - cg[lo], ct[i + 1] - ct[lo]
        with np.errstate(divide="ignore", invalid="ignore"): return np.where(t >= min_events, (1 - g / np.maximum(t, 1)) / (1 - objective), 0.0)
    return (burn(long_min) >= factor) & (burn(short_min) >= factor)

def alert_rule(name, objective, threshold_sli, factor, long_w, short_w, severity, dashboard_uid, panel_id):
    def expr(w):
        sel = 'service_name="supportops-agent"'
        ratio = f'sum(rate(supportops_agent_requests_total{{{sel},supportops_status=~"ok|escalated"}}[{w}])) / sum(rate(supportops_agent_requests_total{{{sel}}}[{w}]))'
        return f"(1 - ({ratio})) / {1 - objective:.6g}"
    q = lambda ref, w: {"refId": ref, "relativeTimeRange": {"from": 600, "to": 0}, "datasourceUid": "prometheus", "model": {"refId": ref, "expr": expr(w), "instant": True, "intervalMs": 15000, "maxDataPoints": 43200}}
    red = lambda ref, src: {"refId": ref, "datasourceUid": "__expr__", "model": {"refId": ref, "type": "reduce", "expression": src, "reducer": "last", "settings": {"mode": "dropNN"}}}
    return {"uid": f"slo-{name}-{severity}", "title": f"SLO {name} · {severity} (×{factor:g})", "condition": "F", "for": "0s" if severity == "page" else "1m", "noDataState": "OK", "execErrState": "Error", "isPaused": False,
            "data": [q("A", long_w), q("B", short_w), red("C", "A"), red("D", "B"),
                     {"refId": "E", "datasourceUid": "__expr__", "model": {"refId": "E", "type": "math", "expression": f"$C > {factor:g} && $D > {factor:g}"}},
                     {"refId": "F", "datasourceUid": "__expr__", "model": {"refId": "F", "type": "threshold", "expression": "E", "conditions": [{"evaluator": {"type": "gt", "params": [0]}}]}}],
            "annotations": {"__dashboardUid__": dashboard_uid, "__panelId__": str(panel_id), "summary": f"SLO {name}: burn rate >= {factor:g}x en {long_w} y {short_w}", "runbook_url": f"https://example.com/runbooks/slo-{name}"},
            "labels": {"severity": severity, "service": "supportops-agent", "slo": name}}

def provision_group(rules, interval):
    if not interval.endswith("s") or not interval[:-1].isdigit() or int(interval[:-1]) % 10: raise ValueError("el intervalo debe ser múltiplo de 10 s")
    return {"apiVersion": 1, "groups": [{"orgId": 1, "name": "supportops-slo", "folder": "Curso observabilidad de agentes", "interval": interval, "rules": rules}]}

def incident_report(good, total, fire, start_min, end_min, objective):
    idx = np.where(fire)[0]; first = int(idx[idx >= start_min][0]); last = int(idx[-1])
    bad = int((total[start_min:end_min] - good[start_min:end_min]).sum())
    return {"ttd_min": first - start_min, "reset_min": last - end_min, "bad_tickets": bad, "budget_month_pct": 100 * bad / ((1 - objective) * total.sum() * 30),
            "false_minutes": int(fire[: max(start_min - 5, 0)].sum() + fire[end_min + 60:].sum())}

def write_postmortem(report):
    hh = lambda m: f"{int(m) // 60:02d}:{int(m) % 60:02d}"
    det = O0 + report["ttd_min"]; pct = round(report["budget_month_pct"]); bad = report["bad_tickets"]
    return {"title": "El proveedor de LLM devuelve 503 durante 45 minutos", "severity": "SEV-2", "summary": f"Entre las 10:00 y las 10:45 el proveedor del LLM falló de forma persistente. {bad} tickets terminaron mal; la alerta de quema rápida saltó a las {hh(det)} (TTD {report['ttd_min']} min) y se consumió el {pct} % del presupuesto mensual.",
            "impact": [f"{bad} tickets malos en 45 minutos.", f"{pct} % del presupuesto mensual de disponibilidad consumido."],
            "timeline": [("10:00", "Empiezan los 503 del proveedor."), (hh(O0 + 4), "El burn rate de 5 min supera 14,4×."), (hh(det), f"Salta la alerta de quema rápida (TTD {report['ttd_min']} min)."), (hh(det + 3), "La guardia reconoce la alerta y abre el dashboard."),
                         (hh(det + 8), "Las trazas muestran 503 en todos los `chat`: causa externa."), ("10:45", "El proveedor se recupera.")],
            "root_cause": "Degradación del proveedor de LLM sin proveedor de reserva ni modo degradado automático en el sistema.",
            "contributing": ["No existía conmutación a un modelo de reserva.", "El runbook no describía el procedimiento de conmutación."],
            "went_well": ["La alerta multiventana no dio falsas alarmas y se resolvió sola."], "went_wrong": [f"El TTD de {report['ttd_min']} min dejó clientes sin respuesta."], "lucky": ["Terminó antes del máximo de tráfico del mediodía."],
            "actions": [("Proveedor/modelo de reserva con conmutación automática", "plataforma", "3 semanas", "mitigar"), ("Sonda sintética por minuto del camino LLM", "observabilidad", "2 semanas", "detectar"),
                        ("Actualizar el runbook con el procedimiento de conmutación", "guardia", "1 semana", "prevenir"), ("Plantilla de comunicación a clientes", "soporte", "2 semanas", "mitigar")]}

display(run_rubric())